In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import importlib

from kfirlib.comparisons import *
from kfirlib.util import *
from kfirlib import gpt
from IPython import display

In [ ]:
ROOT = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\Included studies'
OUT_DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables'

In [ ]:
pdf_paths = []
for current_dir, subdirs, files in os.walk(ROOT):
    for file in files:
        path = os.path.join(current_dir, file)
        if path.endswith('.pdf'):
            pdf_paths.append(path)

pdf_paths

In [ ]:
client = gpt._CLIENT

In [ ]:
files_with_paths = []
for path in tqdm(pdf_paths):
    files_with_paths.append({'path': path,
        'file_object': client.files.create(
        file=open(path, "rb"),
        purpose="user_data")})


In [ ]:
len(files_with_paths)

In [ ]:
SYSTEM_PROMPT = f'''
You are a scientist. You will be given several PDF files of scientific articles. You need to answer the user's question based on *all* the articles 
you are given.
'''
QUESTION = "What is the prevalence and pattern of post-COVID-19 symptoms and symptom clusters up to 12 months after infection, and how do they vary across time periods and between symptom clusters in observational studies of confirmed SARS-CoV-2 patients?"

In [ ]:
def run_model_on_batch(batch):
    content = [
                {
                    "type": "input_text",
                    "text": QUESTION,
                },]
    for item in batch:
        content.append({ "type": "input_file", "file_id": item['file_object'].id,})
    response = client.responses.create(
        model=gpt.Models.MINI,
        input=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": content,
            }
        ]
    )
    return response.output_text

In [ ]:
OUTPUTS = None
TO_PROCESS = None

In [ ]:
def run_on_missing_items(items, batch_size):
    global OUTPUTS, TO_PROCESS
    to_process = [item for item in items if 'title' not in item]
    TO_PROCESS = to_process
    print(f'Running on {len(to_process)} articles')

    outputs = []
    batches = list(batched(to_process, batch_size))
    for batch in tqdm(batches):
        outputs.append(run_model_on_batch(batch))

    OUTPUTS = outputs
    missing_items = 0
    updated_items = 0

    for i, (output, batch) in enumerate(zip(outputs, batches)):
        try:
            res = parse_pdf_response(output)
        except Exception as e:
            print(f'Error in batch {i}: {e}')
            missing_items += len(batch)
            continue
        if len(res) != len(batch):
            print(f'Batch {i}: Expected {len(batch)} results but got {len(res)}.')
            missing_items += len(batch)
            continue
        for item, result in zip(batch, res):
            item.update(result)
        updated_items += len(batch)

    print(f'Updated {updated_items} items, missing {missing_items} items.')


In [ ]:
def run_in_batches(items, batch_size):
    print(f'Running on {len(items)} articles')

    outputs = []
    batches = list(batched(items, batch_size))
    for batch in tqdm(batches):
        outputs.append(run_model_on_batch(batch))

    return outputs


In [ ]:
results = run_in_batches(files_with_paths, 6)

In [ ]:
print('\n\n'.join(results))

In [ ]:
system_instructions = "You will be asked to summarize several results into one"
joined_results = "\n===============================\'".join(results)
prompt = f"""You will be given multiple answers to the same question. The question is:

{QUESTION}

The answers are:

{joined_results}

Please summarize all the answers into a single answer."""
full_summary = gpt.query(model=gpt.Models.MINI, instructions=system_instructions, input=prompt)

In [ ]:
display.Markdown(full_summary)

In [ ]:
batches = batched(pdf_paths, 6)
rows = []
for batch, response in zip(batches, results):
    rows.append({
        'files': ', '.join(batch),
        'response': response})
rows.append({'files': 'all', 'response': full_summary})
df = pd.DataFrame(rows)
df

In [ ]:
df.to_excel(r'C:\Users\israe\OneDrive\Documents\post doc\kfir\covid\tables\gpt_symptoms_summary.xlsx', index=False)